In [6]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch

load_dotenv()

model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

In [7]:
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    tools=[TavilySearch()],
    middleware=[SummarizationMiddleware(
       trigger= ("messages", 6),
       model=model,
       keep=("messages", 2),
    )
    ],
    checkpointer=InMemorySaver(),
)



In [8]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

result =agent.invoke({"messages": [HumanMessage(content="What is the weather in Bangalore?")]}, config)

In [9]:
result =agent.invoke({"messages": [HumanMessage(content="What should i eat in mumbai?")]}, config)

In [10]:
result =agent.invoke({"messages": [HumanMessage(content="What should i see in mumbai")]}, config)
result

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user’s primary goal is to obtain current weather information for Bangalore and receive food recommendations for Mumbai.\n\n## SUMMARY\n\n- The user asked for the current weather in Bangalore.\n- The AI retrieved multiple weather reports from various sources.\n- The consolidated weather information for Bangalore is: temperature around 29.4°C (85°F), light rain showers, 50% humidity, wind from the northeast at about 3.6 mph (5.8 kph), fully cloudy sky, and a feels-like temperature of 30.2°C (86.4°F).\n- The user then asked for food recommendations in Mumbai.\n\n## ARTIFACTS\n\n- Weather data retrieved from multiple sources including Business Today, Weather Underground, WeatherAPI, Ventusky, AQI.in, Met Office, MSN Weather, IMD, AccuWeather, and The Weather Channel.\n- No files or documents were created or modified.\n\n## NEXT STEPS\n\n- Provide food recommendations or popular di

In [11]:
from IPython.display import Markdown

Markdown(result['messages'][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT

The user’s primary goal is to obtain current weather information for Bangalore and receive food recommendations for Mumbai.

## SUMMARY

- The user asked for the current weather in Bangalore.
- The AI retrieved multiple weather reports from various sources.
- The consolidated weather information for Bangalore is: temperature around 29.4°C (85°F), light rain showers, 50% humidity, wind from the northeast at about 3.6 mph (5.8 kph), fully cloudy sky, and a feels-like temperature of 30.2°C (86.4°F).
- The user then asked for food recommendations in Mumbai.

## ARTIFACTS

- Weather data retrieved from multiple sources including Business Today, Weather Underground, WeatherAPI, Ventusky, AQI.in, Met Office, MSN Weather, IMD, AccuWeather, and The Weather Channel.
- No files or documents were created or modified.

## NEXT STEPS

- Provide food recommendations or popular dishes to eat in Mumbai as requested by the user.

In [12]:
import os
env= dict(os.environ)
env['PATH'] = "C:\\Users\\SivaPrasad\\.local\\bin" + os.pathsep + env['PATH']

In [63]:
from langchain.agents.middleware import ShellToolMiddleware,HostExecutionPolicy, DockerExecutionPolicy, TodoListMiddleware
model= ChatOpenAI(model="gpt-5.4-mini")

agent = create_agent(
    model=model,   
    middleware=[
        TodoListMiddleware(),
        
        ShellToolMiddleware(       
        shell_command="C:\\Program Files\\Git\\bin\\bash.exe",
        workspace_root="C:\\npci-mumbai\\ws\\invoice-utils",
        env=env,
      
    )],
    system_prompt=(
        "You are a release engineer with shell access to a Python project that uses uv. "
       
        "The project root is your working directory."       
        "Report pass/fail for each step and a final release-ready verdict."
    ),
    
)


In [64]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Check whether invoice-utils v0.1.0 is ready to release. "
        ),
    }]
})

result


{'messages': [HumanMessage(content='Check whether invoice-utils v0.1.0 is ready to release. ', additional_kwargs={}, response_metadata={}, id='e7822e71-d0d6-4564-a02a-0b25fcf5ad6c'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 1491, 'total_tokens': 1568, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EU7acj8I3ItOXi6y4JjQ8PUx4GL6R', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0f6d7-9e9d-7f00-8793-9781cdbd3e51-0', tool_calls=[{'name': 'write_todos', 'args': {'todos': [{'content': 'Inspect re

In [73]:
result['messages'][14].tool_calls


[{'name': 'write_todos',
  'args': {'todos': [{'content': 'Inspect repository and identify release checks for invoice-utils v0.1.0',
     'status': 'completed'},
    {'content': 'Run relevant tests/builds/validation for release readiness',
     'status': 'completed'},
    {'content': 'Summarize pass/fail per step and final release verdict',
     'status': 'in_progress'}]},
  'id': 'call_mZhze8L0UlJGeMQw2g8Gw76W',
  'type': 'tool_call'}]